# 多模型 PR / 指标对比绘图

基于各实验目录下的 `pr_curves/pr_metrics.json`（默认可视化为：组合 / 线性基线 / 原型基线三套结果），绘图风格与 `tools/pr/plot_pr_curves.py` 一致。

**用法**：在配置单元格修改 `MODELS`，每项为 `{"name": "图例显示名", "pr_json": ".../pr_metrics.json"}`。

**分场景**：分场景行常仅有 F1；分场景子图使用对应 `*_Category_Eval.per_class` 的宏平均精确率、召回率、F1。


In [40]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# 仓库根：在仓库根或 tools/pr 下运行均可
_here = os.path.abspath(os.getcwd())
if os.path.basename(_here) == "pr":
    _REPO_ROOT = os.path.abspath(os.path.join(_here, "..", ".."))
else:
    _REPO_ROOT = _here
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

from config.matplotlib_font import *

FONT_NAME = "AR PL UMing CN"
FIXED_WIDTH = 30
FIXED_IOU = 0.5
SAVE_DPI = 600

SCENES = [
    "curve",
    "extreme_weather",
    "intersection",
    "night",
    "merge_split_case",
    "updown",
]
SCENE_TITLE = {
    "curve": "弯道",
    "extreme_weather": "极端天气",
    "intersection": "路口",
    "night": "夜间",
    "merge_split_case": "分流合流",
    "updown": "上下坡",
}

# ====== 默认三个实验：各自 output 下仅一份 pr_metrics.json，对应 best 等单次全量 PR 评测 ======
MODELS = [
    {
        "name": "本文",
        "pr_json": os.path.join(
            _REPO_ROOT,
            "output/llanetv1/openlane1000/category/clrnet_combined_resnet34/pr_curves/pr_metrics.json",
        ),
    },
    {
        "name": "线性基线",
        "pr_json": os.path.join(
            _REPO_ROOT,
            "output/llanetv1/openlane1000/category/clrnet_linear_resnet34_base/pr_curves/pr_metrics.json",
        ),
    },
    {
        "name": "原型基线",
        "pr_json": os.path.join(
            _REPO_ROOT,
            "output/llanetv1/openlane1000/category/clrnet_prototype_resnet34_base/pr_curves/pr_metrics.json",
        ),
    },
]

OUT_DIR = os.path.join(_REPO_ROOT, "output/analysis/pr_model_compare")
os.makedirs(OUT_DIR, exist_ok=True)

loaded = [
    {"name": m["name"], "data": json.load(open(m["pr_json"], "r"))} for m in MODELS
]

In [41]:
def macro_prf_from_category_eval(block):
    if not block or "per_class" not in block:
        return None, None, None
    rows = []
    for x in block["per_class"]:
        try:
            cid = int(x.get("class_id", -1))
        except Exception:
            continue
        if cid <= 0:
            continue
        rows.append(x)
    if not rows:
        return None, None, None
    p = float(np.mean([float(x.get("precision", 0.0)) for x in rows]))
    r = float(np.mean([float(x.get("recall", 0.0)) for x in rows]))
    f1 = float(np.mean([float(x.get("f1", 0.0)) for x in rows]))
    return p, r, f1


def series_overall_iou(data):
    rows = [d for d in data if int(d.get("Width", -1)) == FIXED_WIDTH]
    rows.sort(key=lambda x: float(x["IoU_Threshold"]))
    xs = [float(d["IoU_Threshold"]) for d in rows]
    ps = [float(d["Total_Precision"]) for d in rows]
    rs = [float(d["Total_Recall"]) for d in rows]
    fs = [float(d["Total_F1"]) for d in rows]
    return xs, ps, rs, fs


def series_overall_width(data):
    rows = [d for d in data if abs(float(d.get("IoU_Threshold", 0)) - FIXED_IOU) < 1e-6]
    rows.sort(key=lambda x: int(x["Width"]))
    xs = [int(d["Width"]) for d in rows]
    ps = [float(d["Total_Precision"]) for d in rows]
    rs = [float(d["Total_Recall"]) for d in rows]
    fs = [float(d["Total_F1"]) for d in rows]
    return xs, ps, rs, fs


def series_scene_iou(data, scene):
    key = f"{scene}_Category_Eval"
    buf = []
    for d in data:
        if int(d.get("Width", -1)) != FIXED_WIDTH:
            continue
        if key not in d:
            continue
        p, r, f1 = macro_prf_from_category_eval(d.get(key))
        if p is None:
            continue
        buf.append((float(d["IoU_Threshold"]), p, r, f1))
    buf.sort(key=lambda t: t[0])
    if not buf:
        return None
    xs, ps, rs, fs = zip(*buf)
    return list(xs), list(ps), list(rs), list(fs)


def series_scene_width(data, scene):
    key = f"{scene}_Category_Eval"
    buf = []
    for d in data:
        if abs(float(d.get("IoU_Threshold", 0)) - FIXED_IOU) > 1e-6:
            continue
        if key not in d:
            continue
        p, r, f1 = macro_prf_from_category_eval(d.get(key))
        if p is None:
            continue
        buf.append((int(d["Width"]), p, r, f1))
    buf.sort(key=lambda t: t[0])
    if not buf:
        return None
    xs, ps, rs, fs = zip(*buf)
    return list(xs), list(ps), list(rs), list(fs)


def series_cat_f1_iou(data, key_macro, key_weight):
    rows = [d for d in data if int(d.get("Width", -1)) == FIXED_WIDTH]
    rows.sort(key=lambda x: float(x["IoU_Threshold"]))
    xs = [float(d["IoU_Threshold"]) for d in rows]
    m = [float(d.get(key_macro, 0.0)) for d in rows]
    w = [float(d.get(key_weight, 0.0)) for d in rows]
    return xs, m, w


def series_cat_f1_width(data, key_macro, key_weight):
    rows = [d for d in data if abs(float(d.get("IoU_Threshold", 0)) - FIXED_IOU) < 1e-6]
    rows.sort(key=lambda x: int(x["Width"]))
    xs = [int(d["Width"]) for d in rows]
    m = [float(d.get(key_macro, 0.0)) for d in rows]
    w = [float(d.get(key_weight, 0.0)) for d in rows]
    return xs, m, w


def style_ax(ax, xlabel, ylabel, title):
    ax.set_xlabel(xlabel, fontname=FONT_NAME, fontsize=11)
    ax.set_ylabel(ylabel, fontname=FONT_NAME, fontsize=11)
    ax.set_title(title, fontname=FONT_NAME, fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.35)


def legend_models_colors_linestyles(
    ax,
    colors,
    model_names,
    fontsize=12,
    loc_models="lower left",
    loc_metric="lower right",
):
    """图内双图例：颜色=模型；线型=F1/P/R。"""
    model_hs = [
        plt.Line2D([0], [0], color=colors[i], lw=2.6, label=model_names[i])
        for i in range(len(model_names))
    ]
    metric_hs = [
        plt.Line2D(
            [0],
            [0],
            color="0.25",
            linestyle="-",
            lw=2.6,
            marker="o",
            markersize=6,
            label="F1",
        ),
        plt.Line2D(
            [0],
            [0],
            color="0.25",
            linestyle="--",
            lw=2.6,
            marker="s",
            markersize=6,
            label="P",
        ),
        plt.Line2D(
            [0],
            [0],
            color="0.25",
            linestyle=":",
            lw=2.6,
            marker="^",
            markersize=6,
            label="R",
        ),
    ]

    leg_m = ax.legend(
        handles=model_hs,
        loc=loc_models,
        fontsize=fontsize,
        title="模型",
        framealpha=0.95,
        facecolor="white",
        edgecolor="0.6",
    )
    leg_m.get_title().set_fontname(FONT_NAME)
    leg_m.get_title().set_fontsize(fontsize)
    for t in leg_m.get_texts():
        t.set_fontname(FONT_NAME)
    ax.add_artist(leg_m)

    leg_s = ax.legend(
        handles=metric_hs,
        loc=loc_metric,
        fontsize=fontsize,
        title="线型",
        framealpha=0.95,
        facecolor="white",
        edgecolor="0.6",
    )
    leg_s.get_title().set_fontname(FONT_NAME)
    leg_s.get_title().set_fontsize(fontsize)
    for t in leg_s.get_texts():
        t.set_fontname(FONT_NAME)


def legend_models_colors_linestyles_fig_top(
    fig,
    colors,
    model_names,
    fontsize=12,
    y=0.985,
    x_models=0.30,
    x_metric=0.76,
):
    """整图内顶部区域：模型颜色 + 线型说明（放在图内空白处）。"""
    model_hs = [
        plt.Line2D([0], [0], color=colors[i], lw=2.6, label=model_names[i])
        for i in range(len(model_names))
    ]
    metric_hs = [
        plt.Line2D(
            [0],
            [0],
            color="0.25",
            linestyle="-",
            lw=2.6,
            marker="o",
            markersize=6,
            label="F1",
        ),
        plt.Line2D(
            [0],
            [0],
            color="0.25",
            linestyle="--",
            lw=2.6,
            marker="s",
            markersize=6,
            label="P",
        ),
        plt.Line2D(
            [0],
            [0],
            color="0.25",
            linestyle=":",
            lw=2.6,
            marker="^",
            markersize=6,
            label="R",
        ),
    ]

    n = len(model_names)
    leg_m = fig.legend(
        handles=model_hs,
        loc="upper center",
        bbox_to_anchor=(x_models, y),
        ncol=min(n, 4),
        fontsize=fontsize,
        title="模型",
        framealpha=0.95,
        facecolor="white",
        edgecolor="0.6",
    )
    leg_m.get_title().set_fontname(FONT_NAME)
    leg_m.get_title().set_fontsize(fontsize)
    for t in leg_m.get_texts():
        t.set_fontname(FONT_NAME)
    fig.add_artist(leg_m)

    leg_s = fig.legend(
        handles=metric_hs,
        loc="upper center",
        bbox_to_anchor=(x_metric, y),
        ncol=3,
        fontsize=fontsize,
        title="线型",
        framealpha=0.95,
        facecolor="white",
        edgecolor="0.6",
    )
    leg_s.get_title().set_fontname(FONT_NAME)
    leg_s.get_title().set_fontsize(fontsize)
    for t in leg_s.get_texts():
        t.set_fontname(FONT_NAME)

In [ ]:
def plot_overall_total_prf_compare(
    model_entries,
    out_path,
    # ===== Figure & spacing (overall 1×2) =====
    FIGSIZE=(14.0, 5.2),
    LEFT=0.07,
    RIGHT=0.98,
    TOP=0.90,
    BOTTOM=0.12,
    WSPACE=0.25,
    # ===== Fonts =====
    FS_TITLE=12,
    FS_LABEL=11,
    FS_TICK=10,
    FS_LEGEND=12,
    FS_LEGEND_TITLE=12,
    # ===== Legend placement (整图坐标；(0,0)=左上, (1,1)=右下) =====
    LEG_MODELS_XY=(0.62, 0.18),
    LEG_METRIC_XY=(0.82, 0.18),
    LEG_MODELS_NCOL=1,
    LEG_FRAME_ALPHA=0.95,
):
    """overall_total_prf_compare.png：左 IoU（扩展宽度固定）右扩展宽度（IoU 固定），P/R/F1。"""

    def _xy_topleft_to_mpl(xy):
        x, y = xy
        return (x, 1 - y)

    n = len(model_entries)
    cmap = plt.get_cmap("tab10")
    colors = [cmap(i % 10) for i in range(n)]
    names = [ent["name"] for ent in model_entries]

    fig, (axL, axR) = plt.subplots(1, 2, figsize=FIGSIZE)
    fig.subplots_adjust(left=LEFT, right=RIGHT, top=TOP, bottom=BOTTOM, wspace=WSPACE)

    for i, ent in enumerate(model_entries):
        data = ent["data"]
        c = colors[i]

        xi, pi, ri, fi = series_overall_iou(data)
        xw, pw, rw, fw = series_overall_width(data)

        axL.plot(xi, fi, color=c, linestyle="-", marker="o", markersize=3)
        axL.plot(xi, pi, color=c, linestyle="--", marker="s", markersize=3)
        axL.plot(xi, ri, color=c, linestyle=":", marker="^", markersize=3)

        axR.plot(xw, fw, color=c, linestyle="-", marker="o", markersize=3)
        axR.plot(xw, pw, color=c, linestyle="--", marker="s", markersize=3)
        axR.plot(xw, rw, color=c, linestyle=":", marker="^", markersize=3)

    # axes style（本图独立参数）
    for ax, title, xlabel in [
        (axL, f"P/R/F1 （随IoU，扩展宽度={FIXED_WIDTH}px）", "IoU 阈值"),
        (axR, f"P/R/F1 （随扩展宽度，IoU={FIXED_IOU}）", "扩展宽度"),
    ]:
        ax.set_title(title, fontname=FONT_NAME, fontsize=FS_TITLE)
        ax.set_xlabel(xlabel, fontname=FONT_NAME, fontsize=FS_LABEL)
        ax.set_ylabel("指标值", fontname=FONT_NAME, fontsize=FS_LABEL)
        ax.tick_params(axis="both", labelsize=FS_TICK)
        ax.grid(True, linestyle="--", alpha=0.35)

    # legends（整图坐标；放在图内空白）
    model_handles = [
        plt.Line2D([0], [0], color=colors[i], lw=2.6, label=names[i]) for i in range(n)
    ]
    metric_handles = [
        plt.Line2D(
            [0],
            [0],
            color="0.25",
            linestyle="-",
            lw=2.6,
            marker="o",
            markersize=6,
            label="F1",
        ),
        plt.Line2D(
            [0],
            [0],
            color="0.25",
            linestyle="--",
            lw=2.6,
            marker="s",
            markersize=6,
            label="P",
        ),
        plt.Line2D(
            [0],
            [0],
            color="0.25",
            linestyle=":",
            lw=2.6,
            marker="^",
            markersize=6,
            label="R",
        ),
    ]

    leg_m = fig.legend(
        handles=model_handles,
        loc="upper left",
        bbox_to_anchor=_xy_topleft_to_mpl(LEG_MODELS_XY),
        ncol=LEG_MODELS_NCOL,
        fontsize=FS_LEGEND,
        framealpha=LEG_FRAME_ALPHA,
        facecolor="white",
        edgecolor="0.6",
    )
    for t in leg_m.get_texts():
        t.set_fontname(FONT_NAME)
    fig.add_artist(leg_m)

    leg_s = fig.legend(
        handles=metric_handles,
        loc="upper left",
        bbox_to_anchor=_xy_topleft_to_mpl(LEG_METRIC_XY),
        ncol=3,
        fontsize=FS_LEGEND,
        framealpha=LEG_FRAME_ALPHA,
        facecolor="white",
        edgecolor="0.6",
    )
    for t in leg_s.get_texts():
        t.set_fontname(FONT_NAME)

    fig.savefig(out_path, dpi=SAVE_DPI)
    plt.close(fig)
    print("saved", out_path)

In [43]:
def plot_scenes_3x2_macro_prf(
    model_entries,
    out_path,
    # ===== Figure & spacing (outer 3×2 + inner 1×2) =====
    FIGSIZE=(20.0, 13.2),
    GS_LEFT=0.035,
    GS_RIGHT=0.997,
    GS_TOP=0.975,
    GS_BOTTOM=0.060,
    GS_HSPACE=0.55,
    GS_WSPACE=0.18,
    SUB_WSPACE=0.10,
    # ===== Fonts =====
    FS_TITLE=12,
    FS_LABEL=11,
    FS_TICK=10,
    FS_LEGEND=12,
    FS_LEGEND_TITLE=12,
    # ===== Legend placement (整图坐标；(0,0)=左上, (1,1)=右下) =====
    LEG_MODELS_XY=(0.18, 0.02),
    LEG_METRIC_XY=(0.62, 0.02),
    LEG_MODELS_NCOL=3,
    LEG_FRAME_ALPHA=0.95,
    # ===== Misc =====
    DROP_YLABEL_ON_RIGHT_COL=True,
):
    """scenes_3x2_macro_prf.png：3×2 场景；每格左 IoU、右扩展宽度；宏平均 P/R/F1。"""

    def _xy_topleft_to_mpl(xy):
        x, y = xy
        return (x, 1 - y)

    n = len(model_entries)
    cmap = plt.get_cmap("tab10")
    colors = [cmap(i % 10) for i in range(n)]
    names = [ent["name"] for ent in model_entries]

    fig = plt.figure(figsize=FIGSIZE, constrained_layout=False)
    gs = GridSpec(3, 2, figure=fig)
    gs.update(
        left=GS_LEFT,
        right=GS_RIGHT,
        top=GS_TOP,
        bottom=GS_BOTTOM,
        hspace=GS_HSPACE,
        wspace=GS_WSPACE,
    )

    for idx, scene in enumerate(SCENES):
        r, ccol = divmod(idx, 2)
        sg = gs[r, ccol].subgridspec(1, 2, wspace=SUB_WSPACE)
        ax_iou = fig.add_subplot(sg[0, 0])
        ax_w = fig.add_subplot(sg[0, 1])

        for i, ent in enumerate(model_entries):
            col = colors[i]
            si = series_scene_iou(ent["data"], scene)
            sw = series_scene_width(ent["data"], scene)
            if si:
                xi, pi, ri, fi = si
                ax_iou.plot(xi, fi, color=col, linestyle="-", marker="o", markersize=2)
                ax_iou.plot(xi, pi, color=col, linestyle="--", marker="s", markersize=2)
                ax_iou.plot(xi, ri, color=col, linestyle=":", marker="^", markersize=2)
            if sw:
                xw, pw, rw, fw = sw
                ax_w.plot(xw, fw, color=col, linestyle="-", marker="o", markersize=2)
                ax_w.plot(xw, pw, color=col, linestyle="--", marker="s", markersize=2)
                ax_w.plot(xw, rw, color=col, linestyle=":", marker="^", markersize=2)

        title = SCENE_TITLE.get(scene, scene)

        for ax, sub_title, xlabel in [
            (ax_iou, f"{title}（随 IoU）", "IoU 阈值"),
            (ax_w, f"{title}（随扩展宽度）", "扩展宽度"),
        ]:
            ax.set_title(sub_title, fontname=FONT_NAME, fontsize=FS_TITLE)
            # 仅最底行显示 x label 和 x 轴刻度；前两行全部隐藏
            if r == 2:
                ax.set_xlabel(xlabel, fontname=FONT_NAME, fontsize=FS_LABEL)
                ax.tick_params(axis="x", labelbottom=True)
            else:
                ax.set_xlabel("")
                ax.tick_params(axis="x", labelbottom=False)
            ax.set_ylabel("宏平均 P/R/F1", fontname=FONT_NAME, fontsize=FS_LABEL)
            ax.tick_params(axis="both", labelsize=FS_TICK)
            ax.grid(True, linestyle="--", alpha=0.35)

        # 右侧（Width）子图通常与相邻列的文字/刻度冲突，统一去掉 y 轴标题
        ax_w.set_ylabel("")
        if DROP_YLABEL_ON_RIGHT_COL and ccol == 1:
            ax_iou.set_ylabel("")

    # legends（整图坐标；放在图内空白）
    model_handles = [
        plt.Line2D([0], [0], color=colors[i], lw=2.6, label=names[i]) for i in range(n)
    ]
    metric_handles = [
        plt.Line2D(
            [0],
            [0],
            color="0.25",
            linestyle="-",
            lw=2.6,
            marker="o",
            markersize=6,
            label="F1",
        ),
        plt.Line2D(
            [0],
            [0],
            color="0.25",
            linestyle="--",
            lw=2.6,
            marker="s",
            markersize=6,
            label="P",
        ),
        plt.Line2D(
            [0],
            [0],
            color="0.25",
            linestyle=":",
            lw=2.6,
            marker="^",
            markersize=6,
            label="R",
        ),
    ]

    leg_m = fig.legend(
        handles=model_handles,
        loc="upper left",
        bbox_to_anchor=_xy_topleft_to_mpl(LEG_MODELS_XY),
        ncol=LEG_MODELS_NCOL,
        fontsize=FS_LEGEND,
        framealpha=LEG_FRAME_ALPHA,
        facecolor="white",
        edgecolor="0.6",
    )
    for t in leg_m.get_texts():
        t.set_fontname(FONT_NAME)
    fig.add_artist(leg_m)

    leg_s = fig.legend(
        handles=metric_handles,
        loc="upper left",
        bbox_to_anchor=_xy_topleft_to_mpl(LEG_METRIC_XY),
        ncol=3,
        fontsize=FS_LEGEND,
        framealpha=LEG_FRAME_ALPHA,
        facecolor="white",
        edgecolor="0.6",
    )
    for t in leg_s.get_texts():
        t.set_fontname(FONT_NAME)

    fig.savefig(out_path, dpi=SAVE_DPI)
    plt.close(fig)
    print("saved", out_path)

In [44]:
def plot_cat_mf1_wf1_2x2(
    model_entries,
    out_path,
    # ===== Figure & spacing (2×2) =====
    FIGSIZE=(13.0, 9.0),
    LEFT=0.08,
    RIGHT=0.98,
    TOP=0.92,
    BOTTOM=0.10,
    WSPACE=0.20,
    HSPACE=0.28,
    # ===== Fonts =====
    FS_TITLE=12,
    FS_LABEL=11,
    FS_TICK=10,
    FS_LEGEND=12,
    FS_LEGEND_TITLE=12,
    # ===== Legend placement (整图坐标；(0,0)=左上, (1,1)=右下) =====
    USE_FIGURE_LEGEND=False,
    LEG_MODELS_XY=(0.20, 0.02),
    LEG_MODELS_NCOL=3,
    LEG_FRAME_ALPHA=0.95,
):
    """cat_mf1_wf1_2x2.png：2×2（mF1/wF1，IoU/扩展宽度）。"""

    def _xy_topleft_to_mpl(xy):
        x, y = xy
        return (x, 1 - y)

    n = len(model_entries)
    cmap = plt.get_cmap("tab10")
    colors = [cmap(i % 10) for i in range(n)]

    fig, axes = plt.subplots(2, 2, figsize=FIGSIZE)
    fig.subplots_adjust(
        left=LEFT,
        right=RIGHT,
        top=TOP,
        bottom=BOTTOM,
        wspace=WSPACE,
        hspace=HSPACE,
    )

    km, kw = "Cat_F1_Macro", "Cat_F1_Weighted"

    for i, ent in enumerate(model_entries):
        c = colors[i]
        name = ent["name"]
        data = ent["data"]
        xi, mi, wi = series_cat_f1_iou(data, km, kw)
        xw, mw, ww = series_cat_f1_width(data, km, kw)
        axes[0, 0].plot(xi, mi, color=c, marker="o", markersize=3, label=name)
        axes[0, 1].plot(xw, mw, color=c, marker="s", markersize=3, label=name)
        axes[1, 0].plot(xi, wi, color=c, marker="o", markersize=3, label=name)
        axes[1, 1].plot(xw, ww, color=c, marker="s", markersize=3, label=name)

    titles = [
        (0, 0, f"mF1 随 IoU（扩展宽度={FIXED_WIDTH}）", "IoU 阈值", "mF1"),
        (0, 1, f"mF1 随扩展宽度（IoU={FIXED_IOU}）", "扩展宽度", "mF1"),
        (1, 0, f"wF1 随 IoU（扩展宽度={FIXED_WIDTH}）", "IoU 阈值", "wF1"),
        (1, 1, f"wF1 随扩展宽度（IoU={FIXED_IOU}）", "扩展宽度", "wF1"),
    ]
    for r, c, ttl, xl, yl in titles:
        ax = axes[r, c]
        ax.set_title(ttl, fontname=FONT_NAME, fontsize=FS_TITLE)
        ax.set_xlabel(xl, fontname=FONT_NAME, fontsize=FS_LABEL)
        ax.set_ylabel(yl, fontname=FONT_NAME, fontsize=FS_LABEL)
        ax.tick_params(axis="both", labelsize=FS_TICK)
        ax.grid(True, linestyle="--", alpha=0.35)

    if USE_FIGURE_LEGEND:
        names = [ent["name"] for ent in model_entries]
        model_handles = [
            plt.Line2D([0], [0], color=colors[i], lw=2.6, label=names[i])
            for i in range(n)
        ]
        leg = fig.legend(
            handles=model_handles,
            loc="upper left",
            bbox_to_anchor=_xy_topleft_to_mpl(LEG_MODELS_XY),
            ncol=LEG_MODELS_NCOL,
            fontsize=FS_LEGEND,
            framealpha=LEG_FRAME_ALPHA,
            facecolor="white",
            edgecolor="0.6",
        )
        for t in leg.get_texts():
            t.set_fontname(FONT_NAME)
    else:
        for ax in axes.ravel():
            leg = ax.legend(fontsize=FS_LEGEND, loc="best", framealpha=0.95)
            if leg:
                for t in leg.get_texts():
                    t.set_fontname(FONT_NAME)

    fig.savefig(out_path, dpi=SAVE_DPI)
    plt.close(fig)
    print("saved", out_path)

In [45]:
# 运行：图 1/3（整体）
PATH_OUT = os.path.join(OUT_DIR, "overall_total_prf_compare.png")

CFG_OVERALL = dict(
    FIGSIZE=(14.0, 5.2),
    LEFT=0.07,
    RIGHT=0.98,
    TOP=0.90,
    BOTTOM=0.14,
    WSPACE=0.25,
    FS_TITLE=20,
    FS_LABEL=20,
    FS_TICK=20,
    FS_LEGEND=15,
    FS_LEGEND_TITLE=15,
    LEG_MODELS_XY=(0.35, 0.10),
    LEG_METRIC_XY=(0.57, 0.10),
    LEG_MODELS_NCOL=1,
    LEG_FRAME_ALPHA=0.95,
)

plot_overall_total_prf_compare(loaded, PATH_OUT, **CFG_OVERALL)

saved /data1/lxy_log/workspace/ms/UnLanedet/output/analysis/pr_model_compare/overall_total_prf_compare.png


In [31]:
# 运行：图 2/3（六分场景 3×2）
PATH_OUT = os.path.join(OUT_DIR, "scenes_3x2_macro_prf.png")

CFG_SCENES = dict(
    FIGSIZE=(22.0, 13.2),
    GS_LEFT=0.060,
    GS_RIGHT=0.995,
    GS_TOP=0.955,
    GS_BOTTOM=0.060,
    GS_HSPACE=0.20,
    GS_WSPACE=0.10,
    SUB_WSPACE=0.20,
    FS_TITLE=20,
    FS_LABEL=20,
    FS_TICK=20,
    FS_LEGEND=20,
    FS_LEGEND_TITLE=20,
    LEG_MODELS_XY=(0.32, 0.037),
    LEG_METRIC_XY=(0.80, 0.037),
    LEG_MODELS_NCOL=3,
    LEG_FRAME_ALPHA=0.95,
    DROP_YLABEL_ON_RIGHT_COL=True,
)

plot_scenes_3x2_macro_prf(loaded, PATH_OUT, **CFG_SCENES)

saved /data1/lxy_log/workspace/ms/UnLanedet/output/analysis/pr_model_compare/scenes_3x2_macro_prf.png


In [39]:
# 运行：图 3/3（mF1 / wF1 2×2）
PATH_OUT = os.path.join(OUT_DIR, "cat_mf1_wf1_2x2.png")

CFG_CAT = dict(
    FIGSIZE=(13.0, 9.0),
    LEFT=0.06,
    RIGHT=0.98,
    TOP=0.95,
    BOTTOM=0.09,
    WSPACE=0.20,
    HSPACE=0.35,
    FS_TITLE=20,
    FS_LABEL=20,
    FS_TICK=20,
    FS_LEGEND=15,
    FS_LEGEND_TITLE=15,
    USE_FIGURE_LEGEND=True,
    LEG_MODELS_XY=(0.04, 0.035),
    LEG_MODELS_NCOL=3,
    LEG_FRAME_ALPHA=0.95,
)

plot_cat_mf1_wf1_2x2(loaded, PATH_OUT, **CFG_CAT)

saved /data1/lxy_log/workspace/ms/UnLanedet/output/analysis/pr_model_compare/cat_mf1_wf1_2x2.png
